# 📌 Chapter 2

## Turning a text into tokens

First of all, we're gonna import the Frankenstein text so that it can serve as context for our tokenizer.
It is from this text that we will create a vocabulary.

In [ ]:
import requests

url = "https://raw.githubusercontent.com/lucasaraujomedeiros/llm-lab/main/frankenstein.txt"
file_path = "frankenstein.txt"

response = requests.get(url, timeout=30)
response.raise_for_status()

with open(file_path, "wb") as f:
    f.write(response.content)

In [ ]:
with open("frankenstein.txt", encoding="utf-8") as f:
    raw_text = f.read()



- Using the regex library, we break the text into smaller tokens


In [ ]:
import re
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
len(preprocessed)

- Enumerating the tokens and making a vocabulary

In [ ]:
all_words = sorted(set(preprocessed))
len(all_words)

In [ ]:
all_words[:100]

In [ ]:
vocab = {s:i for i,s in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
  print(item)
  if i >= 99:
    break

- Adding special tokens
  - EOS: End of Sequence
  - UNK: To represent words that are not included in the vocabulary

In [ ]:
all_words.extend(["[EOS]", "[UNK]"])
vocab = {s:i for i,s in enumerate(all_words)}
len(vocab)

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

- Creating a class for the tokenizer

In [ ]:
class myTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        text_preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        text_preprocessed = [item.strip()  for item in text_preprocessed if item.strip()]
        text_preprocessed = [item if item in self.str_to_int else "[UNK]" for item in text_preprocessed]

        idx = [self.str_to_int[s] for s in text_preprocessed]
        return idx

    def decode(self, idx):
        text = " ".join([self.int_to_str[i] for i in idx])
        text = re.sub(r'\s+([;,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = myTokenizer(vocab)
text = "Hello, do you like tea? [EOS] In the sunlit terraces of the palace."

tokenizer.encode(text)

In [ ]:
tokenizer.decode(tokenizer.encode(text))

## Byte Pair Encoding

- BPE is a way to tokenize text that breaks unknown words into subwords that can be tokenized by the model. For example: The word strangestfeeling, will probably be divided into tokens like ["strang", "est", "feel", "ing"].

In [ ]:
import importlib
import tiktoken

In [ ]:
text = "Hello, do you like tea? [EOS] In the sunlit terraces of the someunknownpalace."

tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.encode(text, allowed_special={"[EOS]"})

In [ ]:
tokenizer.decode(tokenizer.encode(text, allowed_special={"[EOS]"}))

In [ ]:
enc_text = tokenizer.encode(raw_text, allowed_special={"[EOS]"})
print(len(enc_text))

- Creating a Dataset that tokenizes the text and splits it into fixed-length sequences.

In [ ]:
import torch

In [ ]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"[EOF]"})
        assert len(token_ids) > max_length

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

- Creating a DataLoader to iterate over the dataset and create batches.

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

- Creating an embedding layer that maps each token in the vocabulary to a vector of size 256.

In [ ]:
vocab_size = tokenizer.n_vocab
output_dim = 256

embendding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [ ]:
token_embeddings = embendding_layer(inputs)
token_embeddings.shape

- Creating a positional embedding layer to represent the position of each token in the sequence.

In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
pos_embeddings.shape

In [ ]:
input_embeddings = token_embeddings + pos_embeddings
input_embeddings.shape